# Diagnóstico inicial dos dados

Vou começar esta análise fazendo uma leitura geral da base consolidada. A ideia aqui ainda não é tirar conclusões sobre gasto público, mas entender com calma o que existe nos dados: quais anos aparecem, quantas capitais existem em cada ano, quais estágios da despesa estão disponíveis e como as contas foram classificadas.

Para isso, vou usar o arquivo Parquet gerado pelo pipeline em `dados_processados/finbra_consolidado.parquet`.

Também vou manter os valores monetários armazenados como números. O ajuste feito aqui é apenas na exibição do notebook, para usar o padrão brasileiro, com ponto como separador de milhar e vírgula como separador decimal.

In [1]:
from pathlib import Path

import pandas as pd

def formatar_numero_br(valor):
    return f"{valor:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", formatar_numero_br)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dados_processados").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

BASE_PATH = PROJECT_ROOT / "dados_processados" / "finbra_consolidado.parquet"
BASE_PATH.relative_to(PROJECT_ROOT)

PosixPath('dados_processados/finbra_consolidado.parquet')

In [2]:
df = pd.read_parquet(BASE_PATH)
df.head()

,ano,Instituição,Cod.IBGE,UF,População,Coluna,Conta,tipo_conta,codigo_conta,nome_conta,codigo_funcao,nome_funcao,codigo_subfuncao,nome_subfuncao,Identificador da Conta,Valor
0,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,Despesas Exceto Intraorçamentárias,total,<NA>,Despesas Exceto Intraorçamentárias,<NA>,<NA>,<NA>,<NA>,siconfi-cor_TotalDespesas,"874.885.274,98"
1,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01 - Legislativa,função,01,Legislativa,01,Legislativa,<NA>,<NA>,siconfi-cor_TotalDespesas,"29.014.059,54"
2,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54"
3,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03 - Essencial à Justiça,função,03,Essencial à Justiça,03,Essencial à Justiça,<NA>,<NA>,siconfi-cor_TotalDespesas,"18.822.895,07"
4,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,03.092 - Representação Judicial e Extrajudicial,subfunção,03.092,Representação Judicial e Extrajudicial,03,Essencial à Justiça,092,Representação Judicial e Extrajudicial,siconfi-cor_TotalDespesas,"18.822.895,07"


In [8]:
small_df = df[df['codigo_subfuncao'] == '031']
small_df.head()

,ano,Instituição,Cod.IBGE,UF,População,Coluna,Conta,tipo_conta,codigo_conta,nome_conta,codigo_funcao,nome_funcao,codigo_subfuncao,nome_subfuncao,Identificador da Conta,Valor
2,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Empenhadas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"29.014.059,54"
72,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Liquidadas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"28.264.216,80"
140,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Despesas Pagas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"28.264.216,80"
208,2020,Prefeitura Municipal de Rio Branco - AC,1200401,AC,407319,Inscrição de Restos a Pagar Não Processados,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"749.842,74"
556,2020,Prefeitura Municipal de Porto Alegre - RS,4314902,RS,1483771,Despesas Empenhadas,01.031 - Ação Legislativa,subfunção,01.031,Ação Legislativa,01,Legislativa,031,Ação Legislativa,siconfi-cor_TotalDespesas,"97.555.360,50"


In [9]:
small_df.info()

<class 'pandas.DataFrame'>
Index: 561 entries, 2 to 50142
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   ano                     561 non-null    int64  
 1   Instituição             561 non-null    string 
 2   Cod.IBGE                561 non-null    string 
 3   UF                      561 non-null    string 
 4   População               561 non-null    Int64  
 5   Coluna                  561 non-null    string 
 6   Conta                   561 non-null    string 
 7   tipo_conta              561 non-null    string 
 8   codigo_conta            561 non-null    string 
 9   nome_conta              561 non-null    string 
 10  codigo_funcao           561 non-null    string 
 11  nome_funcao             561 non-null    string 
 12  codigo_subfuncao        561 non-null    string 
 13  nome_subfuncao          561 non-null    string 
 14  Identificador da Conta  561 non-null    string 
 15  Val

## Tamanho e estrutura da base

Primeiro, vou conferir o tamanho geral da base, os tipos das colunas e a presença de valores nulos. Isso vai me ajudar a separar problemas reais de dados de nulos que fazem sentido pela própria classificação criada no pipeline.

In [3]:
resumo_estrutura = pd.DataFrame(
    {
        "métrica": ["linhas", "colunas", "anos", "capitais"],
        "valor": [
            len(df),
            len(df.columns),
            df["ano"].nunique(),
            df["Cod.IBGE"].nunique(),
        ],
    }
)

resumo_estrutura

,métrica,valor
0,linhas,50334
1,colunas,16
2,anos,6
3,capitais,26


In [10]:
resumo_colunas = pd.DataFrame(
    {
        "coluna": df.columns,
        "tipo": df.dtypes.astype(str).values,
        "nulos": df.isna().sum().values,
        "nulos_%": (df.isna().mean().values * 100).round(2),
    }
)

resumo_colunas

,coluna,tipo,nulos,nulos_%
0,ano,int64,0,"0,00"
1,Instituição,string,0,"0,00"
2,Cod.IBGE,string,0,"0,00"
3,UF,string,0,"0,00"
4,População,Int64,0,"0,00"
5,Coluna,string,0,"0,00"
6,Conta,string,0,"0,00"
7,tipo_conta,string,0,"0,00"
8,codigo_conta,string,1304,"2,59"
9,nome_conta,string,0,"0,00"


Depois de olhar os nulos, já dá para separar uma coisa importante: alguns deles são esperados. Linhas do tipo `total`, por exemplo, não pertencem a uma função específica, e linhas que não são subfunção não têm `codigo_subfuncao` nem `nome_subfuncao`.

## Anos disponíveis e completude

Antes de comparar evolução no tempo, preciso conferir quantas capitais aparecem em cada ano. Essa checagem é central, porque o README já alerta que 2025 está incompleto e eu não quero misturar anos completos com um ano parcial sem perceber.

In [5]:
resumo_anos = (
    df.groupby("ano", as_index=False)
    .agg(
        linhas=("ano", "size"),
        capitais=("Cod.IBGE", "nunique"),
        ufs=("UF", "nunique"),
        valor_total=("Valor", "sum"),
    )
    .assign(ano_parcial=lambda dados: dados["capitais"] < 26)
)

resumo_anos

,ano,linhas,capitais,ufs,valor_total,ano_parcial
0,2020,8988,26,26,"1.643.749.022.172,56",False
1,2021,8867,26,26,"1.784.544.669.315,28",False
2,2022,9312,26,26,"2.116.930.276.072,95",False
3,2023,9576,26,26,"2.453.736.945.780,88",False
4,2024,9528,26,26,"2.813.142.903.138,99",False
5,2025,4063,11,11,"1.125.836.501.748,72",True


In [6]:
capitais_por_ano = (
    df[["ano", "Instituição", "Cod.IBGE", "UF"]]
    .drop_duplicates()
    .sort_values(["ano", "UF", "Instituição"])
)

capitais_por_ano.groupby("ano")["Instituição"].nunique()

ano
2020    26
2021    26
2022    26
2023    26
2024    26
2025    11
Name: Instituição, dtype: int64

Essa contagem já confirma o principal cuidado temporal da base: 2020 a 2024 têm 26 capitais, enquanto 2025 aparece com menos capitais. Então, nas comparações históricas, vou tratar 2025 como ano parcial.

## Estágios da despesa

Agora quero olhar a coluna `Coluna`, que identifica o estágio da despesa. Nas próximas análises, o foco vai ser comparar `Despesas Empenhadas` e `Despesas Pagas`, mas antes disso vale conferir todos os estágios que aparecem na base.

In [7]:
resumo_estagios = (
    df.groupby("Coluna", as_index=False)
    .agg(
        linhas=("Coluna", "size"),
        valor_total=("Valor", "sum"),
        anos=("ano", "nunique"),
    )
    .sort_values("linhas", ascending=False)
)

resumo_estagios

,Coluna,linhas,valor_total,anos
0,Despesas Empenhadas,11495,"4.058.888.797.973,94",6
1,Despesas Liquidadas,11435,"3.823.937.287.644,98",6
2,Despesas Pagas,11403,"3.735.695.439.925,60",6
3,Inscrição de Restos a Pagar Não Processados,8310,"231.594.623.309,91",6
4,Inscrição de Restos a Pagar Processados,7691,"87.824.169.374,95",6


## Tipos de conta

Em seguida, vou conferir a classificação criada para a coluna `Conta`. Essa etapa é importante porque a base original mistura funções, subfunções e totais; se eu somar tudo sem separar, posso contar o mesmo valor mais de uma vez.

In [8]:
resumo_tipos_conta = (
    df.groupby("tipo_conta", as_index=False)
    .agg(
        linhas=("tipo_conta", "size"),
        contas_distintas=("Conta", "nunique"),
        valor_total=("Valor", "sum"),
    )
    .sort_values("linhas", ascending=False)
)

resumo_tipos_conta

,tipo_conta,linhas,contas_distintas,valor_total
2,subfunção,30122,121,"3.515.162.328.541,62"
1,função,12813,27,"3.858.983.027.805,87"
0,demais_subfunções,6095,26,"343.820.699.264,25"
3,total,1304,2,"4.219.974.262.617,64"


In [9]:
exemplos_por_tipo = (
    df[["tipo_conta", "Conta", "codigo_conta", "nome_conta", "codigo_funcao", "nome_funcao"]]
    .drop_duplicates()
    .sort_values(["tipo_conta", "Conta"])
    .groupby("tipo_conta")
    .head(5)
)

exemplos_por_tipo

,tipo_conta,Conta,codigo_conta,nome_conta,codigo_funcao,nome_funcao
275,demais_subfunções,FU01 - Demais Subfunções,FU01,Demais Subfunções,01,Legislativa
560,demais_subfunções,FU02 - Demais Subfunções,FU02,Demais Subfunções,02,Judiciária
985,demais_subfunções,FU03 - Demais Subfunções,FU03,Demais Subfunções,03,Essencial à Justiça
281,demais_subfunções,FU04 - Demais Subfunções,FU04,Demais Subfunções,04,Administração
35840,demais_subfunções,FU05 - Demais Subfunções,FU05,Demais Subfunções,05,Defesa Nacional
1,função,01 - Legislativa,01,Legislativa,01,Legislativa
558,função,02 - Judiciária,02,Judiciária,02,Judiciária
3,função,03 - Essencial à Justiça,03,Essencial à Justiça,03,Essencial à Justiça
5,função,04 - Administração,04,Administração,04,Administração
2697,função,05 - Defesa Nacional,05,Defesa Nacional,05,Defesa Nacional


Para as análises por função, o filtro mais seguro será `tipo_conta == "função"`. As linhas de `total` e `demais_subfunções` não devem entrar junto sem cuidado, porque elas podem misturar agregações e gerar dupla contagem.

## Funções disponíveis

Antes de avançar para indicadores, também quero conferir quais funções aparecem na base consolidada. A função 21 não é esperada para municípios, então a lista deve ir de 01 a 28 com essa ausência.

In [10]:
funcoes_disponiveis = (
    df.loc[df["tipo_conta"] == "função", ["codigo_funcao", "nome_funcao"]]
    .drop_duplicates()
    .sort_values("codigo_funcao")
    .reset_index(drop=True)
)

funcoes_disponiveis

,codigo_funcao,nome_funcao
0,01,Legislativa
1,02,Judiciária
2,03,Essencial à Justiça
3,04,Administração
4,05,Defesa Nacional
5,06,Segurança Pública
6,07,Relações Exteriores
7,08,Assistência Social
8,09,Previdência Social
9,10,Saúde


## O que levo deste diagnóstico

- A base consolidada já reúne os anos de 2020 a 2025.
- Vou tratar 2025 como ano parcial nas comparações históricas.
- Os estágios necessários para comparar empenhado e pago estão presentes.
- A classificação de contas permite separar funções, subfunções, totais e demais subfunções.
- Na próxima etapa, faz sentido calcular indicadores usando primeiro `tipo_conta == "função"`, principalmente a taxa de execução financeira.